# Does PER3 genotype shift the timing of clock-gene expression in human blood?

**Research question.** Does PER3 genotype (4/4 vs 5/5) shift the time of day at which the clock genes PER2, PER3 and NR1D1 reach peak expression in human blood?

Group project notebook (GEO GSE39445). Generated by `scripts/build_notebook.py` from `notebooks/stages/`. Do not edit it by hand; edit the stage notebooks.

Run the first cell (Colab bootstrap), then *Runtime → Run all*.

**Stages:** 1 Question & objectives · 2 Data acquisition · 3 Preprocessing & QC · 4 EDA · 5 Core analysis · 6 Biological interpretation · 7 Machine learning · 8 Conclusions

In [ ]:
# --- Colab bootstrap: in Colab, clone + install the repo; locally, do nothing ---
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/CKmellow/per3-genotype-clock-gene-timing.git"
BRANCH = "main"  # change to your branch name to test it in Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content/per3-genotype-clock-gene-timing")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    src = str(repo / "src")
else:  # local: find the repo root (works from notebooks/ or notebooks/stages/)
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)
    src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)  # makes `import per3clock` work without a kernel restart

from per3clock import config  # noqa: E402

config.set_seeds()
config.ensure_dirs()
print("per3clock root:", config.ROOT, "| Colab:", IN_COLAB)

# Stage 1 · Research question and objectives

**Research question.** Does PER3 genotype (4/4 vs 5/5) shift the time of day at which the clock genes PER2, PER3 and NR1D1 reach peak expression in human blood?

**Hypothesis.** PER3 5/5 ("early bird") carriers reach peak clock-gene expression earlier in the day than 4/4 ("night owl") carriers.

**Objectives**
1. To estimate the time of peak expression of PER2, PER3 and NR1D1 in each participant.
2. To compare peak timing between the PER3 4/4 and 5/5 genotype groups and estimate the size of any shift.
3. To determine whether any shift reflects a change in the whole internal clock, using melatonin-based circadian phase.
4. To determine whether the results support the hypothesis.

## Background

**PER3 length polymorphism and chronotype.** *PER3* (Period 3) carries a variable-number tandem repeat (VNTR) present as 4 or 5 copies, giving 4/4, 4/5 and 5/5 genotypes. Archer et al. (2003) found that the polymorphism correlated significantly with extreme diurnal preference: *"the longer allele associating with morningness and the shorter allele with eveningness"*. The shorter allele was also strongly associated with delayed sleep phase syndrome. This is why 5/5 carriers are called "early birds" and 4/4 carriers "night owls".

**The molecular clock.** In each cell, CLOCK and ARNTL (BMAL1) drive transcription of the *PER* and *CRY* repressors and of *NR1D1/NR1D2* (REV-ERBα/β). These feed back on the loop and give ~24 h oscillations. Peripheral blood cells keep such rhythms, so the timing of peak expression (the **acrophase**) of PER2, PER3 and NR1D1 can be read from repeated blood samples.

**Dataset.** Möller-Levet et al. (2013) collected 10 whole-blood RNA samples from each of 26 participants during ~40 h of total sleep deprivation. This followed one week of sufficient sleep (control, ~8.5 h) and, in a crossover, one week of insufficient sleep (~5.7 h). They reported 1,855 genes with a circadian expression profile in the control condition, and listed PER1–3, CLOCK, NR1D1 and NR1D2 among the circadian-related genes affected by insufficient sleep. Participants were homozygous for the PER3 VNTR (12 with the shorter allele). The paper does not analyse peak timing by PER3 genotype, which is the gap this project addresses. Each sample also carries a **melatonin-based circadian phase**, so we can separate a shift in clock-gene timing from a shift in the whole internal clock (objective 3).

**Design choices.** The main analysis uses the sufficient-sleep ("Sleep Extension") condition. "Sleep Restriction" is used only as a sensitivity analysis. Primary genes are PER2, PER3 and NR1D1. CLOCK, CRY1, ARNTL, PER1 and NR1D2 are secondary.

In [ ]:
print("Primary genes:  ", ", ".join(config.GENES_PRIMARY))
print("Secondary genes:", ", ".join(config.GENES_SECONDARY))
print("Main condition: ", config.MAIN_CONDITION, "| sensitivity:", config.SENSITIVITY_CONDITION)
print("Period (h):     ", config.PERIOD_H, "| seed:", config.SEED)

## References
- Archer SN, Robilliard DL, Skene DJ, Smits M, Williams A, Arendt J, von Schantz M (2003). A length polymorphism in the circadian clock gene *Per3* is linked to delayed sleep phase syndrome and extreme diurnal preference. *Sleep* 26(4):413–415. doi:10.1093/sleep/26.4.413. PMID 12841365.
- Möller-Levet CS, Archer SN, Bucca G, et al. (2013). Effects of insufficient sleep on circadian rhythmicity and expression amplitude of the human blood transcriptome. *PNAS* 110(12):E1132–E1141. doi:10.1073/pnas.1217154110. PMID 23440187.
- GEO accession GSE39445. https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE39445

### Interpretation
The question is specific (three named genes, one genetic contrast, one outcome: peak time), biological, and testable with public data. GSE39445 has repeated blood samples across the 24-h cycle in genotyped participants, plus melatonin-based phase. Objectives 1–2 test the hypothesis directly. Objective 3 asks whether any difference reflects the whole internal clock (a shift relative to clock time that disappears relative to melatonin phase) or clock-gene timing specifically. Objective 4 brings the evidence together.

# Stage 2 · Data acquisition

All data come from GEO series **GSE39445** (platform **GPL15331**, Agilent-026817 University_Surrey_HumanSleep_44k_v1). Small source files are committed in `data/raw/`. The ~100 MB whole-transcriptome series matrix is downloaded on first use and cached in `data/external/`. Everything is loaded through `per3clock.data`, the same functions used by every later stage. The full data dictionary is in `data/README.md`, and the verification report is `results/tables/data_verification.md`.

| File | What it is |
|---|---|
| `data/raw/GSE39445_sample_metadata.csv` | one row per sample: subject, genotype, condition, time awake, clock time, circadian phase |
| `data/raw/GSE39445_clock_genes.csv` | the same samples × 14 probes for 8 clock genes (log2 expression) |
| `data/raw/GPL15331_probe_annotation.csv` | probe ID → gene symbol, name, Ensembl ID, location |
| `data/raw/GSE39445.txt` | GEO SOFT sample headers (provenance) |
| `data/external/GSE39445_series_matrix.txt.gz` | whole transcriptome: 43,758 probes × 438 samples (downloaded) |

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

## 2.1 Sample metadata

`load_metadata()` reads the metadata CSV, recovers the 11 samples whose subject and genotype were blank (from the sample title and the subject's other samples), parses `clock_time` into decimal hours, and adds `condition_code`, `timepoint` and `is_recovered`.

In [ ]:
meta = data.load_metadata()
print(meta.shape)
display(meta.head())
display(meta.dtypes.to_frame("dtype").T)

| Column | Meaning | Units / values |
|---|---|---|
| `sample_id` | GEO sample accession | GSM… |
| `title` | GEO sample title | `Subject_<id>_<SE/SR>_T<n>` |
| `subject` | participant ID | 26 IDs (AF0004…AF0335) |
| `per3_genotype` | PER3 VNTR genotype | `4/4`, `5/5` |
| `sleep_condition` | prior week of sleep | `Sleep Extension` (sufficient, control), `Sleep Restriction` (6 h/night) |
| `condition_code` | short code | `SE`, `SR` |
| `timepoint`, `timepoint_n` | sample number within the 40-h wake period | T1–T10 / 1–10 |
| `hours_awake` | time since waking | h (7.5, 10.5, …, 34.5) |
| `clock_time` | clock time of blood draw | `H:MM` (no date) |
| `clock_time_h` | same, decimal | h in [0, 24) |
| `circadian_phase_h` | time relative to the participant's plasma-melatonin rhythm (GEO) | h, 3-h steps from −15 to 15 |
| `is_recovered` | subject/genotype recovered from title | bool |

In [ ]:
display(meta.loc[meta.is_recovered, ["sample_id", "title", "subject", "per3_genotype", "condition_code", "timepoint"]])

## 2.2 Clock-gene expression

`load_clock_genes()` returns a long table (sample × gene). **Default: one probe per gene**, the probe with the highest mean expression across all 438 samples (`select_probes`, `collapse="max_mean"`). Averaging would mix in near-background probes. For example, two of PER3's three probes have mean log2 3.9 and 4.2. The rule uses only each probe's overall mean, never time, rhythmicity or genotype, so it cannot bias the later tests. `collapse="mean"` (average probes) and `collapse=None` (probe level) remain available.

In [ ]:
clock = data.load_clock_genes()                  # default: selected probe per gene
probes = data.load_clock_genes(collapse=None)    # every probe
print("long table:", clock.shape, "| probe-level:", probes.shape)
display(clock[["sample_id", "gene", "probe_id", "expression", "n_probes", "subject", "per3_genotype", "condition_code", "timepoint", "clock_time_h"]].head(8))

selection = data.clock_probe_selection()         # also saved as results/tables/selected_probes.csv
display(selection)

`expression` is the GEO quantile-normalised **log2** signal (unitless). Each gene has 1–3 Agilent probes, and their IDs match the GPL15331 annotation and the series matrix. The selected probes are PER2 16821, PER3 25341, NR1D1 33493, CRY1 22736 and NR1D2 38683; CLOCK, ARNTL and PER1 have a single probe. PER3 remains the weakest primary gene (mean log2 5.5 against an array median of 6.7).

## 2.3 Probe annotation (GPL15331)

In [ ]:
annot = data.load_probe_annotation()
print(annot.shape, "| probes with a gene symbol:", annot.GENE_SYMBOL.notna().sum())
display(annot.head(3))
display(annot.dropna(subset=["GENE_SYMBOL"]).head(3))

| Column | Meaning |
|---|---|
| `ID` (index) | integer probe ID (= series-matrix `ID_REF`) |
| `PROBE_NAME` | Agilent probe name, or a control name (e.g. `GE_BrightCorner`) |
| `GENE_SYMBOL`, `GENE_NAME` | HGNC symbol and name (blank for controls) |
| `ENSEMBL_ID` | Ensembl identifier |
| `CHROMOSOMAL_LOCATION` | probe coordinates |

## 2.4 GEO SOFT sample headers (provenance)

In [ ]:
soft = data.load_soft_samples()
print(soft.shape)
display(soft.head(3))
print("Samples storing genotype under 'genotype' instead of 'genotype/variation':", soft["genotype"].notna().sum())

## 2.5 Whole-transcriptome series matrix

Downloaded once (~100 MB, about 30 s in Colab) to `data/external/` and cached. `load_expression_matrix()` checks that there are 438 samples and a plausible number of probes. `collapse_probes_to_genes()` makes a genes × samples matrix, keeping by default the same outcome-blind choice of one probe per gene (highest mean across all samples).

In [ ]:
expr, chars = data.load_expression_matrix(with_characteristics=True)
print("probes x samples:", expr.shape)
print(f"log2 range {expr.values.min():.2f}-{expr.values.max():.2f}, median {pd.Series(expr.values.ravel()).median():.2f}, NaN: {int(expr.isna().sum().sum())}")
genes = data.collapse_probes_to_genes(expr, annot)
print("genes x samples:", genes.shape)
display(chars.head(3))

In [ ]:
# Check: the clock-gene CSV equals the series matrix for all 14 probes x 438 samples
wide = probes.pivot(index="sample_id", columns="probe_id", values="expression")
diff = (expr.loc[wide.columns, wide.index].T - wide).abs().to_numpy().max()
print("max |clock CSV - series matrix| =", diff)
assert diff < 1e-6

## 2.6 Study summary and the shared analysis sets

`analysis_samples()` defines the sample set every later stage uses (issue #10):
- **default**: Sleep Extension, subjects with ≥6 of 10 timepoints (main analysis);
- `strict=True`: the original authors' rule (drop series with 2 consecutive or >2 missing timepoints), as a sensitivity analysis;
- `for_phase_analysis=True`: also drops AF0182 in SE, whose melatonin-phase labels are inconsistent. Use it whenever time is `circadian_phase_h`. AF0182 stays in clock-time analyses because its clock times are real.

In [ ]:
summary = (
    meta.groupby(["per3_genotype", "sleep_condition"])
    .agg(subjects=("subject", "nunique"), samples=("sample_id", "size"))
    .unstack("sleep_condition")
)
display(summary)

tp = meta.pivot_table(index=["per3_genotype", "subject"], columns="condition_code", values="sample_id", aggfunc="count", fill_value=0)
display(tp)

variants = {
    "default (main)": dict(),
    "strict=True (sensitivity)": dict(strict=True),
    "for_phase_analysis=True": dict(for_phase_analysis=True),
    "Sleep Restriction (sensitivity)": dict(condition="Sleep Restriction"),
}
rows = []
for label, kw in variants.items():
    s, excl = data.analysis_samples(**kw, metadata=meta, return_exclusions=True)
    gt = s.drop_duplicates("subject").per3_genotype.value_counts()
    rows.append({"set": label, "subjects": s.subject.nunique(), "4/4": gt.get("4/4", 0),
                 "5/5": gt.get("5/5", 0), "samples": len(s), "excluded": ", ".join(excl.subject)})
display(pd.DataFrame(rows))

_, excl = data.analysis_samples(strict=True, for_phase_analysis=True, metadata=meta, return_exclusions=True)
print("Why subjects are excluded (Sleep Extension, strict + phase):")
display(excl)

### Interpretation

- **Size and design.** 438 blood samples from 26 participants: 12 PER3 4/4 and 14 PER3 5/5, all homozygous, so the comparison is a clean two-group contrast. 221 samples are from the sufficient-sleep (SE) condition and 217 from sleep restriction (SR). Samples were taken every 3 h from 7.5 to 34.5 h awake, which covers more than one full 24-h cycle.
- **Data integrity.** After recovery, every sample has a subject and genotype. The 11 recovered genotypes agree with GEO's own (differently named) genotype field. The clock-gene CSV is identical to the series matrix (max difference 0). The metadata agree with GEO on every field.
- **Shared analysis sets.** Main analysis: 24 participants (11 × 4/4, 13 × 5/5), 221 SE samples. Strict sensitivity set: 20 participants (9 × 4/4, 11 × 5/5), 193 samples. Melatonin-phase analyses: 23 participants (AF0182 dropped), 214 samples. AF0079 and AF0091 have only SR samples, and AF0186 and AF0295 have only SE samples. The genotype groups are small, so later stages should report effect sizes with confidence intervals, not only p-values.
- **One probe per gene.** Using the most highly expressed probe avoids diluting signals with near-background probes (two of PER3's three). The choice is outcome-blind.
- **Things later stages must handle** (details in `results/tables/data_verification.md`):
  1. AF0182 SE T4/T5 have identical circadian phase with irregular spacing, probably an annotation error. AF0182 is excluded from phase analyses only.
  2. PER3 is weakly expressed on this array, so Part 2 should confirm it is above background.
  3. `circadian_phase_h` is binned in 3-h steps. The implied phase-0 clock time averages ~04:10, close to the published melatonin midpoint, but the exact definition is unconfirmed.
- The whole-transcriptome matrix (43,758 probes, ~19.5k genes) is available for the genome-wide rhythmicity scan, the enrichment analysis and the ML stage.

**Can these data answer the question?** Yes, with caveats. The data have everything the question needs: PER3-genotyped participants in two homozygous groups, repeated blood samples spanning more than 24 h (so a peak time can be estimated per person), the three target genes measured on the array, and a melatonin-based phase for objective 3. The limits are a small sample (~12 per genotype, so only fairly large shifts are detectable), 3-hourly sampling and 3-h phase bins (coarse timing), and a single cohort with no independent replication set. These limit precision and power, not validity, and stage 8 must state them.

# Stage 3 · Preprocessing and quality control

*Owner: Part 2 (see issue #2).*

## Minimum evidence (lecturer's brief)
*Brief, stage 3: "Clean, filter, normalize or align the data; check it against biology you know must hold." Minimum evidence: "Before/after counts, a comment on anything suspicious."*

- [ ] Clean / filter / normalize: the data are already quantile-normalised by GEO (limma). Show that array distributions are comparable, and document every filter you add
- [ ] **Before/after counts** for every step (samples, subjects per genotype, probes/genes)
- [ ] **Check against biology you know must hold** (e.g. core clock genes expressed above background in blood, which matters most for PER3, the weakest; clock genes vary with time of day in the expected order; sex-marker genes consistent within each subject)
- [ ] **A comment on anything suspicious** (outlier arrays, AF0182 SE phase labels, recovered samples, late draws)
- [ ] Review the shared defaults (`analysis_samples()`, one-probe-per-gene) and say whether QC supports them
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty.

## Contract
**Inputs:** `data.load_metadata()`, `data.load_expression_matrix()`, `data.load_probe_annotation()`, `data.collapse_probes_to_genes()` (default: one probe per gene, highest mean), `data.analysis_samples(..., return_exclusions=True)`, `results/tables/selected_probes.csv`, `sample_anomalies.csv`, `timepoints_per_subject.csv`.

**Shared defaults (issue #10, `data/README.md` → *Shared defaults*):** main set = SE, ≥6 timepoints (24 subjects, 221 samples). `strict=True` = authors' rule (20 subjects, sensitivity). `for_phase_analysis=True` drops AF0182 SE (23 subjects). These are the single source of truth: if QC justifies a change, change `analysis_samples()` in its own PR (tag Part 1) rather than saving a separate sample list.

**Implement in `per3clock.qc`:** `filter_low_expression`, `sample_qc_table`, `apply_qc`, `qc_counts`. Remove the `xfail` markers in `tests/test_qc.py` once done.

**Outputs (names are fixed):**
- `results/tables/qc_counts.csv`: before/after counts per step
- `results/tables/qc_exclusions.csv`: `sample_id, subject, sleep_condition, reason`
- figures: `qc_expression_density.png`, `qc_sample_correlation.png`, `qc_biology_checks.png` (via `plotting.save_fig`, 300 dpi)
- extra tables: `qc_filter_sensitivity.csv`, `qc_sample_metrics.csv`, `qc_clock_gene_background.csv`, `qc_clock_timing_pooled.csv`, `qc_sex_check.csv`
- `data/processed/expressed_genes_SE.csv`: the expressed-gene list (feature set and enrichment background for Parts 3–4)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from per3clock import config, data, plotting, qc

plotting.set_style()
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
FIG_DPI = 300  # report figures

## 3.1 Load data

Everything comes through `per3clock.data`. The main analysis is **Sleep Extension only** (`analysis_samples()`: 24 subjects, 221 samples). Sleep Restriction is a sensitivity analysis and is never mixed into the main QC.

Before any transformation we check what the matrix already is. GEO says the data were quantile-normalised with limma; if that is true every sample has the same distribution, and re-normalising or re-logging would distort it.

In [ ]:
meta = data.load_metadata()
annot = data.load_probe_annotation()
expr = data.load_expression_matrix()            # probes x 438 samples (downloaded once, ~100 MB)
main = data.analysis_samples(metadata=meta)     # shared default: SE, >= 6 timepoints

gt = main.drop_duplicates("subject").per3_genotype.value_counts()
print(f"series matrix: {expr.shape[0]:,} probes x {expr.shape[1]} samples")
print(f"main set: {len(main)} samples, {main.subject.nunique()} subjects "
      f"({gt['4/4']} x 4/4, {gt['5/5']} x 5/5), conditions: {sorted(main.sleep_condition.unique())}")

chk = qc.check_log2_normalised(expr)
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in chk.items() if k != "quantile_spread"})
display(chk["quantile_spread"].to_frame("max - min across 438 samples").T.round(4))

**Result.** Values run from about 1.1 to 19.5 with median 6.70 and no missing cells, which is a log2 scale (a linear Agilent signal spans ~10⁰–10⁶). Every sample's 5th, 25th, 50th, 75th and 95th percentile agrees to within 0.0004 log2 units: the matrix is already quantile-normalised. **We apply no further transformation or normalisation.**

## 3.2 Probe/gene-level filtering

**Steps.** (1) Keep probes with a gene symbol. (2) One probe per gene, the shared outcome-blind default (`collapse_probes_to_genes`, highest mean over all 438 samples, issue #10; not re-selected here). (3) Remove genes that are not expressed above the array background.

**Threshold rule (chosen before looking at any genotype or time effect).** The array carries 757 negative-control spots (153 `(-)3xSLv` and 604 `DarkCorner`) that measure signal with no transcript. Following the limma user guide's Agilent example, a value counts as expressed if it is at least 10 % brighter than the 95th percentile of the negative controls, i.e. `threshold = Q95(negative controls in the 221 main samples) + log2(1.1)`. A gene is kept if it is above the threshold in **≥ 20 % of the main samples**: about 2 of a subject's 10 three-hourly samples, so a gene expressed only around its daily peak is not removed, and well below the smaller genotype group's share (~45 %), so a gene expressed in one genotype only cannot be filtered out. The rule uses the expression values only, never time, genotype or condition.

The placeholder in the stub (log2 > 6 in 20 % of samples) was **not** used: it would remove PER3, a primary gene (see the sensitivity table).

In [ ]:
has_symbol = annot.GENE_SYMBOL.reindex(expr.index).notna()
genes_all = data.collapse_probes_to_genes(expr, annot)       # one probe per gene, all 438 samples
genes_main = genes_all[main.sample_id]

neg_ids = qc.negative_control_ids(annot).intersection(expr.index)
threshold = qc.background_threshold(expr[main.sample_id], annot)
expressed = qc.filter_low_expression(genes_main, threshold, qc.DEFAULT_MIN_FRAC)

print(f"negative-control probes: {len(neg_ids)}; threshold = log2 {threshold:.3f}")
print(f"probes {len(expr):,} -> with gene symbol {int(has_symbol.sum()):,} -> one per gene "
      f"{len(genes_all):,} -> expressed {len(expressed):,} genes "
      f"({len(genes_all) - len(expressed):,} removed)")

sens = qc.filter_sensitivity(genes_main, min_log2_values=(2.5, threshold, 3.5, 4.0, 5.0, 6.0))
display(sens)

fig = qc.plot_expression_qc(genes_main, main, threshold=threshold,
                            negative_controls=expr.loc[neg_ids, main.sample_id])
plotting.save_fig(fig, "qc_expression_density", dpi=FIG_DPI)

In [ ]:
# All main-analysis clock genes must survive the filter
lost = [g for g in config.CLOCK_GENES if g not in expressed.index]
print("clock genes removed by the filter:", lost or "none")
assert not lost, lost
selected = pd.read_csv(config.TABLES_DIR / "selected_probes.csv")
display(selected[selected.selected][["gene", "probe_id", "probe_name", "mean_log2"]])

**Figure `qc_expression_density`.** (a) The per-sample boxes are identical, which is what quantile normalisation should give, so there is no array with a shifted or compressed distribution. (b) The negative-control signal (black) forms a narrow peak near log2 1.5–2.5. Gene means have a small unexpressed component overlapping it and a broad expressed component centred near 8–9. The threshold sits at the upper edge of the background peak. PER3, the weakest primary gene, has a mean of 5.5: well above background, though below the array median.

## 3.3 Sample-level QC

Metrics per sample, computed on the expressed genes (`qc.sample_qc_table`):
- median and IQR (a check of the normalisation) and missing values;
- Pearson r with the median profile of the **same subject's other samples** (leave-one-out). Between-subject differences are large in blood, so this is the sensitive test for a failed or swapped array;
- Pearson r with the median of all samples;
- the 95th percentile of the negative controls (array background), reported but not used to flag.

**Outlier rule:** a sample is flagged if either correlation, after Fisher's z transform, is more than 3 robust SDs (1.4826 × MAD) below the median of all samples. Flagged samples are **reported, not removed**: dropping samples beyond the agreed sets would need a group decision.

In [ ]:
background = qc.negative_control_p95(expr[main.sample_id], annot)
sample_qc = qc.sample_qc_table(expressed, main, background=background)
metric_cols = ["median_log2", "iqr_log2", "n_missing", "corr_to_subject_median",
               "corr_to_all_median", "neg_ctrl_p95"]
display(sample_qc[metric_cols].describe().T.round(4))

show = ["subject", "per3_genotype", "timepoint", "clock_time", "corr_to_subject_median",
        "corr_to_all_median", "z_subject", "z_all", "neg_ctrl_p95", "outlier_reason"]
print(f"Flagged outliers: {int(sample_qc.outlier.sum())} of {len(sample_qc)}")
display(sample_qc.loc[sample_qc.outlier, show].round(3))
print(f"High-background arrays (negative-control p95, robust z > 3): {int(sample_qc.high_background.sum())}")
display(sample_qc.loc[sample_qc.high_background, show].round(3))

fig = qc.plot_sample_qc(sample_qc)
plotting.save_fig(fig, "qc_sample_correlation", dpi=FIG_DPI)

In [ ]:
# Samples already known to be unusual (data/README.md, Known issues): how do they look in QC?
anomalies = pd.read_csv(config.TABLES_DIR / "sample_anomalies.csv")
recovered = main.loc[main.is_recovered, "sample_id"]
known = pd.concat([anomalies[["sample_id", "issue"]],
                   pd.DataFrame({"sample_id": recovered, "issue": "subject/genotype recovered from title"})])
known = known[known.sample_id.isin(sample_qc.index)]
display(known.merge(sample_qc[show[:4] + ["corr_to_subject_median", "z_subject", "outlier"]],
                    left_on="sample_id", right_index=True).round(3))

**Figure `qc_sample_correlation` and tables.** No array looks failed or swapped. Every sample correlates strongly with its own subject's other samples (median r 0.993, minimum 0.969) and with the all-sample median (minimum 0.962); medians and IQRs are identical across samples and there are no missing values.

- **4 mild statistical outliers** (robust z < −3 on the within-subject correlation): AF0069 T10 (r = 0.969), AF0114 T1 and T6 (0.974 each) and AF0145 T4 (0.972). They form the lower tail of a tight distribution rather than standing apart from it, and their correlation with the all-sample median is ordinary. **Kept** (no agreed rule to drop them); listed in `qc_sample_metrics.csv`. *Proposal for the group, not applied:* Part 3 could check that per-subject fits for AF0069, AF0114 and AF0145 do not change materially without these samples (`qc.apply_qc(meta, expr, drop_outliers=True)`); no subject would fall below 6 timepoints.
- **6 arrays with bright negative-control spots** (95th percentile 4.9–6.3 vs median 2.4): AF0101 T1, AF0186 T5, AF0191 T1 and T2, AF0295 T2, AF0335 T5. Their gene-level correlations are normal (0.979–0.994), so this looks like a local artefact at the control spots rather than a poor hybridisation. Reported, not acted on.
- **Known issues look normal at the expression level:** AF0182 SE T4/T5 (|z| ≤ 1.1), the late draws AF0186 T8 and AF0318 T10, and the 11 recovered samples all have ordinary metrics. AF0182's problem is in its phase labels, not its arrays, and the recovered samples behave like their subject's other samples, which supports the recovery of their subject IDs.

## 3.4 Time-series completeness

The shared sample sets (`analysis_samples`, issue #10) are applied unchanged through `qc.apply_qc`, which returns the kept samples and a per-sample exclusion table with reasons.

In [ ]:
variants = {
    "default (main)": {},
    "strict=True (authors' rule, sensitivity)": {"strict": True},
    "for_phase_analysis=True": {"for_phase_analysis": True},
    "Sleep Restriction (sensitivity)": {"condition": "Sleep Restriction"},
}
sets = {}
for label, kw in variants.items():
    kept, excl = qc.apply_qc(meta, **kw)
    sets[label] = kept
    cond = kw.get("condition", config.MAIN_CONDITION)
    print(f"--- {label}: {len(kept)} samples, {kept.subject.nunique()} subjects")
    display(excl[excl.sleep_condition == cond].groupby(["subject", "reason"]).size()
            .rename("samples excluded").reset_index())

kept_main, exclusions = qc.apply_qc(meta)
assert list(kept_main.sample_id) == list(main.sample_id)   # QC keeps exactly the agreed main set
print("Main-set exclusions by reason:")
display(exclusions.reason.value_counts().to_frame("samples"))

Every subject with Sleep Extension samples already has ≥ 6 SE timepoints, so the default removes no SE sample: all 217 exclusions from the main set are Sleep Restriction samples, set aside by design (AF0079 and AF0091 have only SR samples, which is why the main set has 24 subjects). The authors' stricter rule removes AF0112, AF0182, AF0186 and AF0267 (20 subjects, 193 samples). AF0182 SE stays in clock-time analyses and is removed only when time is `circadian_phase_h` (23 subjects, 214 samples): its T4 (02:25) and T5 (04:25) share phase −3 h, and T4 implies phase 0 two hours earlier than its other samples. Its expression is unremarkable in 3.3, which supports a labelling error rather than a bad array.

## 3.5 Biology sanity checks

Things that must hold if the data and labels are right:
1. **Core clock genes are expressed in blood**, above background. This matters most for PER3, the weakest primary gene on this array.
2. **Clock genes vary with time of day in the expected order**: the repressor arm (PER1–3, NR1D1/2) peaks together, roughly opposite the activator ARNTL. Checked with one cosinor per gene on subject-centred expression, genotypes pooled (`qc.pooled_clock_timing`). This is a description of the data, not the per-subject analysis (Part 3), and no p-value is given because pooled repeated measures are not independent.
3. **Sex markers are consistent within each subject.** Sex is not in the GEO metadata; XIST (expressed from the inactive X) and RPS4Y1 (Y chromosome) are used only to check that no subject's samples disagree, which would suggest a sample swap.

In [ ]:
clock = data.load_clock_genes()                 # one probe per gene (shared default)
clock_main = clock[clock.sample_id.isin(main.sample_id)]

bg_check = qc.clock_gene_background_check(genes_main, threshold)
display(bg_check.round(3))

timing = qc.pooled_clock_timing(clock_main)
timing["acrophase_clock"] = timing.acrophase_h.map(lambda h: f"{int(h):02d}:{int(round(h % 1 * 60)) % 60:02d}")
display(timing.round(3))

sex = qc.infer_sex(genes_main, main)
per_subject = sex.drop_duplicates("subject")[["subject", "per3_genotype", "subject_sex"]]
print("samples inconsistent with their subject:", int((~sex.consistent).sum()))
display(pd.crosstab(per_subject.per3_genotype, per_subject.subject_sex))

fig = qc.plot_biology_checks(clock_main, timing, sex, threshold=threshold, array_median=chk["median"])
plotting.save_fig(fig, "qc_biology_checks", dpi=FIG_DPI)

**Figure `qc_biology_checks`.**

1. **Above background.** All eight clock genes exceed the threshold in 100 % of main samples. PER3 is the weakest (mean log2 5.46, 30th percentile of all genes, minimum 4.21 against a threshold of 3.04): low but clearly expressed, so its rhythm estimates will be noisier than PER2's or NR1D1's, not invalid. CLOCK (34th percentile) and CRY1 (38th) are also low.
2. **Expected order.** Pooled peak times: NR1D1 02:13, NR1D2 04:09, PER2 05:03, PER3 06:03, PER1 06:52, so the repressor arm peaks together in the early morning, NR1D1 first. ARNTL's pooled rhythm is weak (r² 0.03), but its peak (14:57) is roughly opposite the PER/NR1D genes, as the clock's feedback loop predicts. CLOCK (r² 0.00) and CRY1 (r² 0.03) show essentially no pooled rhythm, which supports treating them as secondary. These are pooled descriptions only; Part 3 estimates per-subject peaks with uncertainty.
3. **Sex markers.** Every sample matches its subject (0 inconsistent): 10 XIST-high and 14 RPS4Y1-high subjects, with no sign of a sample swap. The split is 5/6 in the 4/4 group and 5/8 in the 5/5 group, roughly balanced; Part 3 may still want sex as a covariate or check.

## 3.6 Before/after table and outputs

In [ ]:
n_probes, n_symbol, n_genes, n_expr = len(expr), int(has_symbol.sum()), len(genes_all), len(expressed)
counts = qc.qc_counts([
    ("1 GEO series matrix (all samples, all probes)", meta, n_probes),
    ("2 probes with a gene symbol", meta, n_symbol),
    ("3 one probe per gene (shared default)", meta, n_genes),
    ("4 Sleep Extension, >=6 timepoints (main set)", main, n_genes),
    ("5 expression filter (main set)", main, n_expr),
    ("6 sample outliers removed (none: flagged only)", kept_main, n_expr),
    ("sensitivity: strict=True", sets["strict=True (authors' rule, sensitivity)"], n_expr),
    ("sensitivity: for_phase_analysis=True", sets["for_phase_analysis=True"], n_expr),
    ("sensitivity: Sleep Restriction (SE gene set)", sets["Sleep Restriction (sensitivity)"], n_expr),
])
display(counts)

counts.to_csv(config.TABLES_DIR / "qc_counts.csv", index=False)
exclusions.to_csv(config.TABLES_DIR / "qc_exclusions.csv", index=False)
sens.to_csv(config.TABLES_DIR / "qc_filter_sensitivity.csv", index=False)
sample_qc.drop(columns=["title"]).round(5).to_csv(config.TABLES_DIR / "qc_sample_metrics.csv")
bg_check.round(4).to_csv(config.TABLES_DIR / "qc_clock_gene_background.csv")
timing.round(4).to_csv(config.TABLES_DIR / "qc_clock_timing_pooled.csv")
sex.round(3).to_csv(config.TABLES_DIR / "qc_sex_check.csv")

sel = data.select_probes(expr, annot).loc[expressed.index]
expressed_table = pd.DataFrame({
    "gene": expressed.index, "probe_id": sel.probe_id.to_numpy(),
    "mean_log2_SE": expressed.mean(axis=1).round(4).to_numpy(),
    "frac_above_threshold": (expressed > threshold).mean(axis=1).round(4).to_numpy(),
})
expressed_table.to_csv(config.PROCESSED_DIR / "expressed_genes_SE.csv", index=False)
print("saved:", ", ".join(["qc_counts.csv", "qc_exclusions.csv", "qc_filter_sensitivity.csv",
      "qc_sample_metrics.csv", "qc_clock_gene_background.csv", "qc_clock_timing_pooled.csv",
      "qc_sex_check.csv", "data/processed/expressed_genes_SE.csv"]))

## Interpretation

**What was changed, and why (counts in `qc_counts.csv`).**
- *Normalisation:* none. The matrix is already log2 and quantile-normalised (3.1), so re-normalising would only distort it.
- *Features:* 43,758 probes → 30,505 with a gene symbol → 19,541 genes (one probe per gene, the shared default) → **17,528 expressed genes**. The 2,013 genes removed are not above the negative-control background (log2 3.044) in at least 20 % of the 221 main samples. The threshold comes from the array's own negative controls, not from the data we want to test, and it ignores time and genotype. Gene counts change modestly with the settings (16,817–17,929 for 10–50 % of samples at this threshold; `qc_filter_sensitivity.csv`), and no clock gene is lost at any threshold up to log2 5. A cut at log2 6, the stub's placeholder, would have removed PER3.
- *Samples:* 438 → **221** (Sleep Extension, 24 subjects: 11 × 4/4, 13 × 5/5). The 217 Sleep Restriction samples are set aside for the sensitivity analysis. **QC removed no sample.**

**Is anything suspicious left?**
- 4 mild correlation outliers (AF0069 T10, AF0114 T1 and T6, AF0145 T4) and 6 arrays with bright control spots. All are kept and listed, and none looks like a failed or swapped array.
- AF0182 SE's inconsistent phase labels are handled by `for_phase_analysis=True`; its arrays are normal.

**Does the data agree with biology that must hold?** Yes.
- All clock genes are expressed above background, with PER3 lowest.
- PER1–3 and NR1D1/2 peak together in the early morning, roughly opposite ARNTL.
- Sex markers are consistent within every subject.

**Do the shared defaults stand?** QC supports them, so none is proposed for change.
- *One probe per gene:* every selected clock-gene probe is well above background. The two unused PER3 probes average 3.9 and 4.2, near background, which confirms that averaging them in would dilute PER3.
- *≥ 6 timepoints:* this rule removes no SE subject.
- *Sample QC:* nothing justifies dropping a sample.

**What QC cannot show.** Expression QC cannot catch a sample whose *time label* is wrong while its RNA is fine (AF0182 was found from its timing, not its expression). Sex is inferred from two genes, not recorded. The background threshold is a judgement, though the sensitivity table shows it does not drive the result.

**Handed to Parts 3–4:**
- `data/processed/expressed_genes_SE.csv`: the expressed-gene set, for the rhythmicity scan's features and as the enrichment background.
- `qc_sample_metrics.csv`: per-sample QC flags.
- PER3 is the noisiest primary gene, so expect wider confidence intervals for it.


# Stage 4 · Exploratory data analysis

*Owner: Part 2 (see issue #2).*

## Minimum evidence (lecturer's brief)
*Brief, stage 4: "Describe and visualize the data; look for patterns, outliers." Minimum evidence: "at least three informative plots, interpretation."*

- [ ] **At least three informative plots** (more is fine; uninformative plots don't count)
- [ ] **An interpretation for each plot**: what pattern or outlier it shows and why it matters for the question
- [ ] Look explicitly for patterns (time of day, subject, genotype, condition) and outliers
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty.

## Contract
**Inputs:** `data.analysis_samples()` (shared default; use `for_phase_analysis=True` for plots against `circadian_phase_h`), `data.load_clock_genes()` (one probe per gene), expression matrix via `data.collapse_probes_to_genes(data.load_expression_matrix())`.

**Implement in `per3clock.eda`:** `plot_clock_gene_profiles`, `pca_samples`, `plot_pca`, `plot_sample_correlation`.

**Outputs:** figures `eda_clock_gene_profiles.png`, `eda_pca_subject.png`, `eda_pca_timepoint.png`, `eda_sample_correlation.png` (via `plotting.save_fig`, 300 dpi); also `eda_clock_gene_profiles_secondary.png`, `eda_pca_overview.png`, `eda_pca_phase.png`, `eda_pca_condition.png`, and tables `eda_clock_gene_profiles.csv`, `eda_subject_timing.csv`, `eda_pca_variance.csv`, `eda_pca_factor_r2.csv`.

**EDA is descriptive.** Genotype groups are shown as mean ± SE, with no hypothesis tests; those are Part 3's job.

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import config, data, eda, plotting, qc

plotting.set_style()
pd.set_option("display.width", 140)
FIG_DPI = 300  # report figures

meta = data.load_metadata()
main = data.analysis_samples(metadata=meta)                              # 24 subjects, 221 samples
phase_set = data.analysis_samples(metadata=meta, for_phase_analysis=True)  # 23 subjects (no AF0182)
clock = data.load_clock_genes()                                          # one probe per gene
clock_main = clock[clock.sample_id.isin(main.sample_id)]
print(len(main), "main samples;", len(phase_set), "samples for phase analyses")

## 4.1 Clock-gene profiles by genotype

Each panel shows one gene: thin lines are individual subjects, thick lines the genotype mean ± SE across subjects (subjects are the unit; a bin is drawn only if it has ≥ 3 subjects). Expression is **centred within each subject** (subject mean subtracted per gene), which removes between-person level differences so the curves show timing; raw levels are in Stage 3 (`qc_biology_checks`).

- **Top row, clock time.** Sampling started between 14:00 and 18:15 depending on the participant and ran across midnight, so clock time is *unwrapped* (`eda.unwrap_clock_time`) and binned in 3-h windows; grouping by T1–T10 would mix clock times up to 4 h apart.
- **Bottom row, circadian phase** (`circadian_phase_h`, already in 3-h bins). Phase 0 is *probably* the plasma-melatonin midpoint; that is inferred from the data (implied mean 04:10 vs the published 04:15), not confirmed. AF0182 is dropped from this row because of its inconsistent phase labels.

In [ ]:
fig = eda.plot_clock_gene_profiles(clock_main, genes=config.GENES_PRIMARY,
                                   time_col=("clock_time_h", "circadian_phase_h"), center=True)
plotting.save_fig(fig, "eda_clock_gene_profiles", dpi=FIG_DPI)

fig = eda.plot_clock_gene_profiles(clock_main, genes=("CLOCK", "CRY1"),
                                   time_col=("clock_time_h", "circadian_phase_h"), center=True)
plotting.save_fig(fig, "eda_clock_gene_profiles_secondary", dpi=FIG_DPI)

profiles = pd.concat([
    eda.genotype_profile_table(clock_main, config.GENES_PRIMARY + ("CLOCK", "CRY1"), tc, center=True)
    .assign(time_axis=tc) for tc in ("clock_time_h", "circadian_phase_h")])
profiles.round(4).to_csv(config.TABLES_DIR / "eda_clock_gene_profiles.csv", index=False)

# Schedules differ between people: implied wake time and phase-0 (melatonin proxy) clock time
timing = eda.subject_timing_summary(main)
timing.round(3).to_csv(config.TABLES_DIR / "eda_subject_timing.csv")
display(timing.groupby("per3_genotype")[["wake_clock_h", "phase0_clock_h"]]
        .agg(["count", "mean", "std", "min", "max"]).round(2).T)
print("Descriptive only (decimal clock hours; AF0182 has no phase-0 value). No test here: see Part 3.")

**Figure `eda_clock_gene_profiles`.**

- **Rhythms.** PER2 and NR1D1 show clear daily rhythms in nearly every subject. NR1D1 rises through the evening and is highest around 01:00–05:00 clock time (phase −3 to 0 h); PER2 is highest a few hours later (about 04:00–08:00, phase 0 to +3 h); both reach a trough in the afternoon. The same shape appears against circadian phase, so the curves follow the internal clock and are not just a drift across 40 h awake. PER3 has a smaller amplitude (about ±0.2 log2) and noisier individual series, highest around 03:00–08:00 (phase 0 to +3 h). One 4/4 subject has a single large spike at its last sample, which is visible in the plot and should be watched in Part 3.
- **Genotype (descriptive only, no test).** The 4/4 and 5/5 mean curves largely overlap, and their ±SE bands meet or overlap in most bins. On *clock time*, the 5/5 mean reaches its highest bin one bin (3 h) earlier than 4/4 for PER2 and NR1D1, in the hypothesised direction, and in the same bin for PER3. On *circadian phase* that difference disappears (NR1D1, PER3) or reverses (PER2). The highest bin is a crude summary at 3-h resolution, so these are impressions, not estimates.
- **Secondary genes (`eda_clock_gene_profiles_secondary`).** CLOCK is flat on both axes, and CRY1 shows at most a weak, broad rise. This supports keeping them secondary.

**Schedules and melatonin timing differ by genotype (table above, descriptive).** On average the 5/5 participants woke about 1.2 h earlier than the 4/4 participants, and their implied phase-0 clock time (the melatonin proxy) was also about 1.2 h earlier. That fits the pattern in the curves: an earlier 5/5 peak in clock time that disappears when time is measured from each person's own melatonin rhythm. Whether this holds up, and how large it is with a CI, is objectives 2–3 and Part 3's job. With 11 vs 13 subjects only shifts larger than roughly 1–2 h are likely to be detectable.

## 4.2 PCA

PCA of the 221 main samples on the expressed genes from Stage 3 (`qc.expressed_genes`: one probe per gene, background filter), each gene centred, not scaled. How much of each PC is explained by subject, genotype, timepoint and circadian phase is measured with η² (share of the PC's variance explained by the grouping; descriptive, no p-values). Subject is nested in genotype, so genotype's η² can never exceed subject's; with 24 subjects η²(subject) would be ≈ 0.1 by chance alone.

In [ ]:
expr = data.load_expression_matrix()
annot = data.load_probe_annotation()
expressed, threshold = qc.expressed_genes(expr, annot, main.sample_id)
print(f"{len(expressed):,} expressed genes (threshold log2 {threshold:.3f}) x {expressed.shape[1]} samples")

scores, explained = eda.pca_samples(expressed, n_components=10)
r2 = eda.pca_factor_r2(scores.iloc[:, :6], main, explained=explained.iloc[:6])
explained.to_frame().assign(cumulative=explained.cumsum()).round(4).to_csv(config.TABLES_DIR / "eda_pca_variance.csv")
r2.round(4).to_csv(config.TABLES_DIR / "eda_pca_factor_r2.csv")
display(explained.to_frame().assign(cumulative=explained.cumsum()).T.round(3))
display(r2.round(3))

plotting.save_fig(eda.plot_pca(scores, main, "subject", explained=explained), "eda_pca_subject", dpi=FIG_DPI)
plotting.save_fig(eda.plot_pca(scores, main, "timepoint", explained=explained), "eda_pca_timepoint", dpi=FIG_DPI)
fig = eda.plot_pca_grid(scores, main, ("per3_genotype", "subject", "timepoint", "clock_time_h"),
                        explained=explained, suptitle="PCA of the main set (Sleep Extension, 221 samples)")
plotting.save_fig(fig, "eda_pca_overview", dpi=FIG_DPI)
# circadian phase: AF0182 SE has inconsistent phase labels, so its samples are not shown
in_phase = scores.index.isin(phase_set.sample_id)
fig = eda.plot_pca(scores[in_phase], phase_set, "circadian_phase_h", explained=explained,
                   title="PCA coloured by circadian phase (23 subjects; AF0182 not shown)")
plotting.save_fig(fig, "eda_pca_phase", dpi=FIG_DPI)

**Side by side: Sleep Extension and Sleep Restriction.** Not part of the main analysis. One PCA on both default sample sets (SE 221 + SR 217 samples, same expressed genes) shows whether the sleep condition moves samples more than the person does.

In [ ]:
sr = data.analysis_samples("Sleep Restriction", metadata=meta)
both = pd.concat([main, sr], ignore_index=True)
genes_both = data.collapse_probes_to_genes(expr, annot).loc[expressed.index, both.sample_id]
scores_b, explained_b = eda.pca_samples(genes_both, n_components=5)
display(eda.pca_factor_r2(scores_b, both, ("subject", "sleep_condition", "timepoint"),
                          explained=explained_b).round(3))
fig = eda.plot_pca_grid(scores_b, both, ("sleep_condition", "subject"), explained=explained_b,
                        suptitle="Side by side, NOT the main analysis: Sleep Extension + Sleep Restriction (438 samples)")
plotting.save_fig(fig, "eda_pca_condition", dpi=FIG_DPI)

**PCA (`eda_pca_subject`, `eda_pca_timepoint`, `eda_pca_overview`, `eda_pca_phase`).** The first two PCs explain 21.7 % and 11.9 % of the variance, and the first ten 64 %. **Subject dominates.** Subject explains 82 % of PC1 and 91 % of PC2 (η²), and 77 % of the variance across PCs 1–6 (weighted). Samples from the same person cluster together; for example, the low PC2 cluster is one 4/4 subject (AF0010), not a group of bad arrays. **Time of day is a minor source of variation:** timepoint explains about 4 % and circadian phase about 4 % across PCs 1–6, with at most 7 % on any one PC (PC3). That is expected, because only a minority of genes are rhythmic in blood, and it is why Part 3 should model each subject's rhythm within that subject rather than compare raw levels between people. **Genotype** explains about 3 %, essentially what is carried along by subject (subject is nested in genotype), and no genotype separation is visible.

**Side by side with Sleep Restriction (`eda_pca_condition`).** On the 438 samples, subject explains 75 % of the leading PCs and sleep condition 0.3 %. A week of restricted sleep does not move the whole transcriptome as much as the difference between people does; the main and sensitivity sets are comparable in overall structure.

## 4.3 Sample correlation

Pearson correlation between all pairs of main-set samples on the expressed genes, ordered by genotype, subject and timepoint, with colour bars for genotype and subject.

In [ ]:
fig = eda.plot_sample_correlation(expressed, main)
plotting.save_fig(fig, "eda_sample_correlation", dpi=FIG_DPI)

display(eda.correlation_summary(expressed, main).round(4))

**Figure `eda_sample_correlation`.** The heatmap shows bright blocks on the diagonal, one per subject. Pairs of samples from the same person correlate more strongly (median r 0.988) than pairs from different people (0.967–0.968), confirming the PCA. Pairs from *different genotypes* correlate as well as pairs of different people within the same genotype (median 0.966 vs 0.968): there is no genome-wide genotype signature. A few subjects (e.g. AF0010) are slightly less similar to everyone else, which is the same person-level structure seen on PC2, not a sample problem.

## Interpretation

- **What dominates the variation?** The person. Subject explains about 77 % of the leading principal components; time of day (timepoint or circadian phase) explains about 4 %, genotype about 3 % (carried by subject), and sleep condition under 1 %. Any analysis of timing must therefore work *within subjects* (per-subject cosinor fits, as planned for Part 3), not compare raw expression across people.
- **Do the clock genes look rhythmic?** Yes for the primary genes. PER2 and NR1D1 clearly are, and PER3 is too but with a smaller amplitude and more noise. They peak in the early morning, in the expected order (NR1D1 before PER2/PER3), on both clock time and circadian phase. CLOCK and CRY1 show little or no rhythm.
- **Any visible genotype difference?** Nothing large. The genotype mean curves overlap. On clock time the 5/5 group looks up to one 3-h bin earlier for PER2 and NR1D1, matching the 5/5 group's earlier wake times and earlier melatonin timing (both about 1.2 h earlier on average); on circadian phase the curves line up. These are visual, descriptive impressions with no test and no CI.
- **What the EDA does not show.** It cannot say whether genotype shifts peak time: that needs per-subject peak estimates, a between-group comparison with a 95 % CI, and the melatonin-relative analysis (Part 3, objectives 1–3). A null or inconclusive answer there would be consistent with these plots, given ~12 subjects per genotype and 3-h sampling.

# Stage 5 · Core analysis: peak timing by PER3 genotype

*Owner: Part 3 (see issue #3). Skeleton: replace the TODO cells, keep the headings.*

## Minimum evidence (lecturer's brief)
*Brief, stage 5: "Apply the methods that directly answer your question." Minimum evidence: "Results table, key figures, statistical evidence" (example: "with FDR correction").*

- [ ] Methods that directly answer objectives 1–3 (per-subject peak times; 4/4 vs 5/5 comparison; melatonin-relative phase)
- [ ] **Results table** (estimates, effect sizes in hours with 95% CIs, test statistics, p-values)
- [ ] **Key figures** (e.g. acrophase rose plots, forest plot of shifts)
- [ ] **Statistical evidence** for each objective, with assumptions checked
- [ ] **FDR correction** (Benjamini–Hochberg) wherever many tests are run: the genome-wide rhythmicity scan, and across the primary + secondary genes
- [ ] Sensitivity analyses: `strict=True` sample set; Sleep Restriction; probe-mean (`collapse="mean"`) vs selected probe
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty. With ~12 subjects per genotype, a non-significant shift is *inconclusive*, not proof of no shift. Give the CI.

## Contract
**Sample sets (shared defaults, `data/README.md`):**
- objectives 1–2 (clock time): `data.analysis_samples()` → 24 subjects (11 × 4/4, 13 × 5/5), 221 samples
- objective 3 (melatonin phase, `time_col="circadian_phase_h"`): `data.analysis_samples(for_phase_analysis=True)` → 23 subjects (AF0182 SE dropped)
- sensitivity: `strict=True` (20 subjects) and `condition="Sleep Restriction"`

**Inputs:** `data.load_clock_genes()` (one probe per gene; `collapse="mean"` for the probe sensitivity check), expression matrix via `data.collapse_probes_to_genes(data.load_expression_matrix())`.

**Implement in `per3clock.rhythm`:** `fit_subject_cosinors`, `rhythmicity_scan` (`cosinor_fit` exists: extend and validate it). **In `per3clock.circstats`:** `circular_mean_h`, `circular_diff_h`, `watson_williams_test`, `permutation_test_acrophase`, `bootstrap_ci_diff_h`.

**Outputs:**
- `results/tables/acrophase_by_subject.csv`: subject × gene × condition × time base (clock time / circadian phase)
- `results/tables/acrophase_genotype_tests.csv`: gene, time base, condition, sample set, mean acrophase 4/4 and 5/5, shift (h, 5/5 − 4/4), 95% CI, permutation p, Watson–Williams p, BH q
- `results/tables/rhythmic_genes_SE.csv`: genome-wide scan with q-values (**input to Part 4**)
- figures: `acrophase_rose_PER2.png` / `_PER3` / `_NR1D1`, `acrophase_shift_forest.png`

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data, plotting

plotting.set_style()

## 5.1 Per-subject cosinor fits (objective 1)

In [43]:
# Load the agreed primary analysis sample (Sleep Extension)
analysis_meta = data.analysis_samples()

# Load the expression values for the selected clock genes
clock_long = data.load_clock_genes()

# Keep only samples included in the primary analysis
selected_sample_ids = analysis_meta["sample_id"].unique()

clock_long = clock_long[
    clock_long["sample_id"].isin(selected_sample_ids)
].copy()

# Check that the expected genes and participants are present
print("Participants:", analysis_meta["subject"].nunique())
print("Samples:", analysis_meta["sample_id"].nunique())
print("Genes:", sorted(clock_long["gene"].unique()))

display(clock_long.head())

Participants: 24
Samples: 221
Genes: ['ARNTL', 'CLOCK', 'CRY1', 'NR1D1', 'NR1D2', 'PER1', 'PER2', 'PER3']


,sample_id,gene,probe_id,expression,n_probes,title,subject,per3_genotype,sleep_condition,condition_code,timepoint,timepoint_n,hours_awake,clock_time,clock_time_h,circadian_phase_h,is_recovered
0,GSM968833,ARNTL,9332,10.065636,1,Subject_AF0004_SE_T1,AF0004,4/4,Sleep Extension,SE,T1,1,7.5,18:15,18.25,-12.0,False
1,GSM968833,CLOCK,26609,5.857082,1,Subject_AF0004_SE_T1,AF0004,4/4,Sleep Extension,SE,T1,1,7.5,18:15,18.25,-12.0,False
2,GSM968833,CRY1,22736,6.192062,2,Subject_AF0004_SE_T1,AF0004,4/4,Sleep Extension,SE,T1,1,7.5,18:15,18.25,-12.0,False
3,GSM968833,NR1D1,33493,8.376711,2,Subject_AF0004_SE_T1,AF0004,4/4,Sleep Extension,SE,T1,1,7.5,18:15,18.25,-12.0,False
4,GSM968833,NR1D2,38683,8.618083,2,Subject_AF0004_SE_T1,AF0004,4/4,Sleep Extension,SE,T1,1,7.5,18:15,18.25,-12.0,False


**5.1.2: Fit the cosinor models**
Now we'll fit each participant's expression pattern twice:
- Clock time: Uses the actual time of day when each sample was collected.
- Circadian phase: Uses the participant's internal biological timing, relative to their circadian phase measurement.

In [44]:
from per3clock import rhythm

# Focus on the three genes in our research objectives
genes_of_interest = ["PER2", "PER3", "NR1D1"]

primary_clock = clock_long[
    clock_long["gene"].isin(genes_of_interest)
].copy()

# Fit one cosinor model per participant and gene using clock time
clock_fits = rhythm.fit_subject_cosinors(
    primary_clock,
    time_col="clock_time_h",
    genes=genes_of_interest,
    period=24.0,
    min_points=4
)

# Fit the same models using circadian phase
phase_fits = rhythm.fit_subject_cosinors(
    primary_clock,
    time_col="circadian_phase_h",
    genes=genes_of_interest,
    period=24.0,
    min_points=4
)

# Label the time basis used by each model
clock_fits["time_base"] = "clock_time"
phase_fits["time_base"] = "circadian_phase"

# Combine both sets of results
acrophase_by_subject = pd.concat(
    [clock_fits, phase_fits],
    ignore_index=True
)

# Check the results
print("Total cosinor fits:", len(acrophase_by_subject))
print("Participants:", acrophase_by_subject["subject"].nunique())
print("Genes:", sorted(acrophase_by_subject["gene"].unique()))
print("\nFits by gene and time basis:")

display(
    acrophase_by_subject.groupby(
        ["gene", "time_base"]
    ).size().rename("number_of_fits").reset_index()
)

display(acrophase_by_subject.head())

Total cosinor fits: 144
Participants: 24
Genes: ['NR1D1', 'PER2', 'PER3']

Fits by gene and time basis:


,gene,time_base,number_of_fits
0,NR1D1,circadian_phase,24
1,NR1D1,clock_time,24
2,PER2,circadian_phase,24
3,PER2,clock_time,24
4,PER3,circadian_phase,24
5,PER3,clock_time,24


,subject,per3_genotype,sleep_condition,gene,mesor,amplitude,acrophase_h,r2,p,n,time_base
0,AF0004,4/4,Sleep Extension,PER2,8.460845,0.102244,9.367660,0.343913,0.228752,10,clock_time
1,AF0004,4/4,Sleep Extension,PER3,5.987668,0.165082,19.956140,0.041966,0.860664,10,clock_time
2,AF0004,4/4,Sleep Extension,NR1D1,8.420100,0.338901,5.136733,0.706374,0.013718,10,clock_time
3,AF0010,4/4,Sleep Extension,PER2,8.801501,0.043126,6.763980,0.174864,0.561793,9,clock_time
4,AF0010,4/4,Sleep Extension,PER3,5.355728,0.178238,3.713953,0.082457,0.772466,9,clock_time


**5.1.3: Save partcipant level results**

In [45]:
from pathlib import Path

# Recreate the clock-time fits using the primary 24-participant sample
clock_fits = rhythm.fit_subject_cosinors(
    primary_clock,
    time_col="clock_time_h",
    genes=genes_of_interest,
    period=24.0,
    min_points=4
)

# Circadian-phase analysis uses the separate 23-participant sample
phase_meta = data.analysis_samples(for_phase_analysis=True)
phase_sample_ids = phase_meta["sample_id"].unique()

phase_long = data.load_clock_genes()
phase_long = phase_long[
    (phase_long["sample_id"].isin(phase_sample_ids)) &
    (phase_long["gene"].isin(genes_of_interest))
].copy()

phase_fits = rhythm.fit_subject_cosinors(
    phase_long,
    time_col="circadian_phase_h",
    genes=genes_of_interest,
    period=24.0,
    min_points=4
)

# Label and combine the two time bases
clock_fits["time_base"] = "clock_time"
phase_fits["time_base"] = "circadian_phase"

acrophase_by_subject = pd.concat(
    [clock_fits, phase_fits],
    ignore_index=True
)

# Save the results
output_dir = Path("results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "acrophase_by_subject.csv"
acrophase_by_subject.to_csv(output_file, index=False)

# Verify the saved analysis
print("Saved to:", output_file)
print("Total fits:", len(acrophase_by_subject))
print("\nFits by time basis and gene:")

display(
    acrophase_by_subject.groupby(
        ["time_base", "gene"]
    ).size().rename("number_of_fits").reset_index()
)

Saved to: results/tables/acrophase_by_subject.csv
Total fits: 141

Fits by time basis and gene:


,time_base,gene,number_of_fits
0,circadian_phase,NR1D1,23
1,circadian_phase,PER2,23
2,circadian_phase,PER3,23
3,clock_time,NR1D1,24
4,clock_time,PER2,24
5,clock_time,PER3,24


## 5.2 Genotype comparison in clock time (objective 2)

In [46]:
from per3clock import circstats
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd

# Store the comparison results
comparison_results = []

# Analyse each gene separately under each time basis
for (gene, time_base), group in acrophase_by_subject.groupby(
    ["gene", "time_base"]
):

    # Separate participants according to genotype
    group_44 = group.loc[
        group["per3_genotype"] == "4/4", "acrophase_h"
    ].dropna().to_numpy()

    group_55 = group.loc[
        group["per3_genotype"] == "5/5", "acrophase_h"
    ].dropna().to_numpy()

    # Remove any non-finite values
    group_44 = group_44[np.isfinite(group_44)]
    group_55 = group_55[np.isfinite(group_55)]

    # Calculate circular mean peak times
    mean_44 = circstats.circular_mean_h(group_44)
    mean_55 = circstats.circular_mean_h(group_55)

    # Calculate the genotype shift: 5/5 minus 4/4
    shift = circstats.circular_diff_h(mean_55, mean_44)

    # Permutation test
    permutation = circstats.permutation_test_acrophase(
        group_55,
        group_44,
        n_perm=10_000
    )

    # Bootstrap 95% confidence interval
    ci_lower, ci_upper = circstats.bootstrap_ci_diff_h(
        group_55,
        group_44,
        n_boot=10_000,
        alpha=0.05
    )

    # Watson-Williams test
    ww = circstats.watson_williams_test(
        group_44,
        group_55
    )

    comparison_results.append({
        "gene": gene,
        "time_base": time_base,
        "condition": "Sleep Extension",
        "sample_set": "Primary",
        "n_44": len(group_44),
        "n_55": len(group_55),
        "mean_acrophase_44": mean_44,
        "mean_acrophase_55": mean_55,
        "shift_h": shift,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "permutation_p": permutation["p"],
        "watson_williams_p": ww["p"]
    })

# Create the results table
genotype_tests = pd.DataFrame(comparison_results)

# Apply Benjamini-Hochberg correction across all six comparisons
genotype_tests["permutation_q"] = multipletests(
    genotype_tests["permutation_p"],
    method="fdr_bh"
)[1]

genotype_tests["watson_williams_q"] = multipletests(
    genotype_tests["watson_williams_p"],
    method="fdr_bh"
)[1]

# Display the results
display(
    genotype_tests.round(4)
)

,gene,time_base,condition,sample_set,n_44,n_55,mean_acrophase_44,mean_acrophase_55,shift_h,ci_lower,ci_upper,permutation_p,watson_williams_p,permutation_q,watson_williams_q
0,NR1D1,circadian_phase,Sleep Extension,Primary,11,12,22.5106,22.8808,0.3703,-1.1650,1.6830,0.6177,0.6284,0.6587,0.6639
1,NR1D1,clock_time,Sleep Extension,Primary,11,13,3.2190,2.0576,-1.1613,-3.0874,0.6766,0.2385,0.2539,0.6587,0.6639
2,PER2,circadian_phase,Sleep Extension,Primary,11,12,0.7637,1.8276,1.0640,-0.4677,2.6932,0.2169,0.2261,0.6587,0.6639
3,PER2,clock_time,Sleep Extension,Primary,11,13,5.4852,4.9794,-0.5057,-2.5659,1.7148,0.6587,0.6639,0.6587,0.6639
4,PER3,circadian_phase,Sleep Extension,Primary,11,12,2.3608,1.1771,-1.1837,-5.2644,2.2261,0.5244,0.5546,0.6587,0.6639
5,PER3,clock_time,Sleep Extension,Primary,11,13,6.4683,5.1100,-1.3583,-6.2774,2.9514,0.5769,0.5740,0.6587,0.6639


## 5.3 Melatonin-relative phase (objective 3)

In [48]:
# Extract only the circadian-phase comparisons
phase_results = genotype_tests[
    genotype_tests["time_base"] == "circadian_phase"
].copy()

# Display the results
display(
    phase_results[
        [
            "gene",
            "n_44",
            "n_55",
            "mean_acrophase_44",
            "mean_acrophase_55",
            "shift_h",
            "ci_lower",
            "ci_upper",
            "permutation_p",
            "permutation_q"
        ]
    ].round(4)
)

,gene,n_44,n_55,mean_acrophase_44,mean_acrophase_55,shift_h,ci_lower,ci_upper,permutation_p,permutation_q
0,NR1D1,11,12,22.5106,22.8808,0.3703,-1.1650,1.6830,0.6177,0.6587
2,PER2,11,12,0.7637,1.8276,1.0640,-0.4677,2.6932,0.2169,0.6587
4,PER3,11,12,2.3608,1.1771,-1.1837,-5.2644,2.2261,0.5244,0.6587


Save comparison results

In [49]:
from pathlib import Path

output_dir = Path("results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

phase_results.to_csv(
    output_dir / "acrophase_circadian_phase_tests.csv",
    index=False
)

print("Circadian-phase comparison results saved.")
display(phase_results)

Circadian-phase comparison results saved.


,gene,time_base,condition,sample_set,n_44,n_55,mean_acrophase_44,mean_acrophase_55,shift_h,ci_lower,ci_upper,permutation_p,watson_williams_p,permutation_q,watson_williams_q
0,NR1D1,circadian_phase,Sleep Extension,Primary,11,12,22.510587,22.880843,0.370256,-1.165030,1.682992,0.6177,0.628429,0.6587,0.663944
2,PER2,circadian_phase,Sleep Extension,Primary,11,12,0.763657,1.827628,1.063971,-0.467673,2.693203,0.2169,0.226084,0.6587,0.663944
4,PER3,circadian_phase,Sleep Extension,Primary,11,12,2.360814,1.177072,-1.183741,-5.264420,2.226061,0.5244,0.554592,0.6587,0.663944


## 5.4 Sensitivity: Sleep Restriction

In [ ]:
# TODO(Part 3): repeat 5.1-5.3 with strict=True, on Sleep Restriction, and with collapse='mean' probes

## 5.5 Genome-wide rhythmicity scan

In [ ]:
# TODO(Part 3): rhythmicity_scan() with BH FDR; save rhythmic_genes_SE.csv

## Interpretation

_TODO(Part 3): for each objective (1–3): the estimate, its uncertainty and what it means. Do 5/5 carriers peak earlier, by how many hours (95% CI), and does the shift remain relative to melatonin phase? Do the sensitivity analyses agree? If the result is null or inconclusive, say so plainly._

# Stage 6 · Biological interpretation

*Owner: Part 4 (see issue #4). Skeleton: replace the TODO cells, keep the headings.*

## Minimum evidence (lecturer's brief)
*Brief, stage 6: "Explain what the results mean biologically." Minimum evidence: "At least one interpretation method, linked back to the question."*

- [ ] **At least one interpretation method** (we plan GO/pathway enrichment, a STRING/NetworkX network and a literature comparison)
- [ ] Each method **explicitly linked back to the research question** (PER3 genotype and the timing of PER2/PER3/NR1D1 peaks)
- [ ] Every citation verified (DOI or PubMed); no invented references
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty.

## Contract
**Inputs:** `results/tables/rhythmic_genes_SE.csv` and `acrophase_genotype_tests.csv` (stage 5). While they don't exist, develop with the 8 clock genes and **don't commit placeholder results**. Background for enrichment: all genes on the array after QC (`data.collapse_probes_to_genes(...)` index).

**Implement in `per3clock.interpret`:** `run_enrichment`, `fetch_string_network`, `build_network`, `network_centrality`. Cache API responses in `data/external/`.

**Outputs:** `results/tables/enrichment_rhythmic.csv`, `results/tables/string_edges.csv`, `results/tables/network_centrality.csv`; figures `enrichment_top_terms.png`, `string_network.png`.

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data, plotting

plotting.set_style()

## 6.1 Enrichment of rhythmic genes

In [ ]:
# TODO(Part 4): run_enrichment() on q<0.05 rhythmic genes vs array background

## 6.2 STRING network of clock genes

In [ ]:
# TODO(Part 4): fetch_string_network(); build_network(); network_centrality(); plot

## 6.3 Literature comparison

In [ ]:
# TODO(Part 4): compare acrophases/shift with published values (verify every citation)

## Interpretation

_TODO(Part 4): what the rhythmic / shifted genes do, how they connect to PER3 in the clock network, how our results compare with the literature, and what that means for the research question._

# Stage 7 · Machine learning: predicting internal circadian phase

*Owner: Part 5 (see issue #5). Skeleton: replace the TODO cells, keep the headings.*

## Minimum evidence (lecturer's brief)
*Brief, stage 7: "Build a model that classifies, predicts or groups your data." Learning outcome 6: "Build and evaluate machine learning models against proper baselines."*

- [ ] A model that **predicts** each sample's internal circadian phase (`circadian_phase_h`) from expression
- [ ] **Evaluated against proper baselines** with leave-one-subject-out CV (no leakage: scaling and feature selection inside folds)
- [ ] Metric: **median absolute circular error in hours**, with the same folds for every model
- [ ] A fairer "added value" comparison: **clock time + genes vs clock time alone**
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty. Matching the clock-time baseline without beating it is an acceptable, reportable result.

## Contract
**Samples:** `data.analysis_samples(for_phase_analysis=True)` (target is `circadian_phase_h`; AF0182 SE dropped; 23 subjects, 214 samples). **Features:** clock genes via `data.load_clock_genes()` (one probe per gene) and/or genes from `data.collapse_probes_to_genes(data.load_expression_matrix())`.

**Target:** `circadian_phase_h` encoded as sin/cos. **CV:** `LeaveOneGroupOut`/`GroupKFold` by `subject`. **Metric:** median absolute circular error (h).
**Baselines:** (a) circular mean phase of the training fold; (b) phase predicted from `clock_time_h` only. Optional: add PER3 genotype as a feature.

**Read this before interpreting (data caveat):** `circadian_phase_h` comes in **3-h steps** and is roughly **clock time minus a per-subject offset** (the clock time of each person's melatonin phase 0, ~02:15–07:25 in SE). So baseline (b) will be **strong**: it is only wrong by that offset. Compare the model with it honestly, using the same LOSO folds and the median absolute circular error. **Matching but not beating the clock-time baseline is an acceptable, reportable result.** The fairer question is whether expression adds information beyond clock time: compare **clock time + genes** against **clock time alone** (e.g. does adding genes reduce the per-subject offset error?). Errors below ~1.5 h are not meaningful with 3-h bins. GSE48113 has no phase labels, so there is no external validation set (`data/README.md`).

**Implement in `per3clock.ml`:** `encode_phase`, `decode_phase`, `circular_abs_error_h`, `loso_cv_predict`, `evaluate_phase_models`.

**Outputs:** `results/tables/ml_phase_prediction.csv` (one row per model, including both baselines and clock time + genes); figures `ml_predicted_vs_true_phase.png`, `ml_error_by_model.png`.

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data, plotting

plotting.set_style()

## 7.1 Features and target

In [ ]:
# TODO(Part 5): build X (genes) and y (sin/cos of circadian_phase_h); groups = subject

## 7.2 Baselines

In [ ]:
# TODO(Part 5): mean-phase and clock-time-only baselines under LOSO

## 7.3 Model

In [ ]:
# TODO(Part 5): e.g. Ridge / ElasticNet / RandomForest in a Pipeline; tune inside folds (nested CV)

## 7.3b Does expression add information beyond clock time?

In [ ]:
# TODO(Part 5): same LOSO folds: clock_time_h only vs clock_time_h + genes; report the difference in median abs circular error (h) with a paired comparison across subjects

## 7.4 Comparison

In [ ]:
# TODO(Part 5): evaluate_phase_models(); table + figures; optional genotype feature

## Interpretation

_TODO(Part 5): does expression predict internal phase better than the baselines, and does it add anything beyond clock time? Which genes carry the signal? If the model only matches the clock-time baseline, say so: that is a valid finding._

# Stage 8 · Conclusions

*Owner: Part 1 (see issue #6). Skeleton: replace the TODO cells, keep the headings.*

## Minimum evidence (lecturer's brief)
*Brief, stage 8: "Answer the research question and state the limitations." Minimum evidence: "One point per objective, plus future steps. Upload to GitHub the code." Learning outcome 7: "Communicate results honestly, including negative or inconclusive findings."*

- [ ] **Answer the research question** in one or two sentences
- [ ] **One point per objective** (1–4), each citing the evidence (table/figure) it rests on
- [ ] **Limitations**
- [ ] **Future steps**
- [ ] Negative or inconclusive findings stated honestly (no overclaiming from small n)
- [ ] Code on GitHub; the final notebook runs top-to-bottom in Google Colab **with outputs saved**

## Contract
**Inputs:** tables in `results/tables/` from stages 3–7 (`qc_counts.csv`, `acrophase_genotype_tests.csv`, `rhythmic_genes_SE.csv`, `enrichment_rhythmic.csv`, `ml_phase_prediction.csv`). Written last, by Part 1, once results exist (issue #6).

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data, plotting

plotting.set_style()

## 8.1 Key numbers

In [ ]:
# TODO(Part 1): load acrophase_genotype_tests.csv, ml_phase_prediction.csv, qc_counts.csv and print the headline results

## Interpretation

_TODO(Part 1): Objective 1: … · Objective 2: … · Objective 3: … · Objective 4 (hypothesis supported?): … · Limitations: small n (24 SE subjects), 3-h sampling and binned phase, homozygotes only, blood not central clock, single dataset · Future steps: …_